# Linear regression: stock daily return from market signals

Part of the ML Portfolio Model Zoo. Educational purposes only.

Predict a stock's daily return (%) from the market's return, its sector's return and unusual trading volume. The data is **synthetic** with a known relationship (market beta about 1.15) plus noise, so the learned coefficients can be checked against the truth.

In [1]:
import pandas as pd
from train import *

# Everything here reuses train.py, so the notebook and the site always use the same pipeline.

## Data

In [2]:
data = make_data()
print(data.shape)
data.head()

(3000, 4)


,market_return_pct,sector_return_pct,volume_ratio,stock_return_pct
0,0.39,1.14,1.08,0.954
1,-1.09,-0.28,0.98,-2.234
2,0.88,1.99,1.35,1.839
3,1.08,-0.32,1.24,0.261
4,-2.10,-1.65,0.35,-2.084


In [3]:
data.describe(include='all').T

,count,mean,std,min,25%,50%,75%,max
market_return_pct,3000.0,0.021963,1.108429,-3.960,-0.710,0.0500,0.75000,3.550
sector_return_pct,3000.0,0.021580,1.040775,-3.620,-0.670,0.0000,0.72000,3.660
volume_ratio,3000.0,1.061910,0.382424,0.290,0.800,0.9900,1.27000,4.000
stock_return_pct,3000.0,0.037891,1.771978,-5.849,-1.173,0.0615,1.25525,5.988


## Train and evaluate

`train_and_evaluate()` fits the scikit-learn pipeline, scores it on a held-out split, exports it to ONNX and asserts that onnxruntime reproduces scikit-learn's predictions.

In [4]:
result = train_and_evaluate()
print(f"R^2 {result.metrics['r2']:.3f}   RMSE {result.metrics['rmse']:.3f}%   MAE {result.metrics['mae']:.3f}%")
print(f"ONNX vs sklearn max abs diff: {result.metrics['onnx_max_abs_diff']:.2e}")

R^2 0.780   RMSE 0.811%   MAE 0.650%
ONNX vs sklearn max abs diff: 5.27e-07


## Predict with the exported ONNX model

In [5]:
example = pd.DataFrame([PRESETS[1]["values"]])[COLUMNS]
pred = onnx_predict(result.model_bytes, example)
print("Sell-off preset -> predicted return", f"{pred[0]:+.2f}%")

Sell-off preset -> predicted return -4.42%


## Site artefacts

Run `uv run python train.py` to write `model.onnx`, `schema.json`, the charts and the .NET parity fixtures into the web app. See the README for details.